#Hw2.1.ipynb

In [ ]:
# Hw2.1

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

url = "https://uk.wikipedia.org/wiki/Коефіцієнт_народжуваності_в_регіонах_України_(1950—2019)"
tables = pd.read_html(url)
birth_df = None
for t in tables:
    if any('2019' in str(c) for c in t.columns):
        birth_df = t
if birth_df is None:
    birth_df = tables[-1]

if birth_df.columns[0] != 'region':
    birth_df = birth_df.rename(columns={birth_df.columns[0]: 'region'})

display(birth_df.head())
print("shape:", birth_df.shape)

birth_df = birth_df.replace("—", np.nan)

display(birth_df.dtypes)

for col in birth_df.columns:
    if col != 'region':
        birth_df[col] = pd.to_numeric(birth_df[col], errors='coerce')

display(birth_df.dtypes)

print("missing share:")
display(birth_df.isnull().mean())

birth_df = birth_df.iloc[:-1, :]

num_cols = birth_df.columns.drop('region')
birth_df[num_cols] = birth_df[num_cols].fillna(birth_df[num_cols].mean())

print("missing share after fill:")
display(birth_df.isnull().mean())

col_2019 = [c for c in birth_df.columns if '2019' in str(c)][0]
avg_2019 = birth_df[col_2019].mean()
regions_above_avg_2019 = birth_df.loc[birth_df[col_2019] > avg_2019, 'region']
print("Регіони > середнього у 2019:", regions_above_avg_2019.tolist())

col_2014 = [c for c in birth_df.columns if '2014' in str(c)][0]
max_2014_region = birth_df.loc[birth_df[col_2014].idxmax(), 'region']
print("Найвища народжуваність у 2014:", max_2014_region)

plt.figure(figsize=(12, 6))
sns.barplot(x='region', y=col_2019, data=birth_df, palette='viridis')
plt.xticks(rotation=90)
plt.title('Народжуваність по регіонах України, 2019')
plt.ylabel('Коефіцієнт народжуваності')
plt.tight_layout()
plt.show()

# Додаткові графіки

top10_2019 = birth_df.sort_values(col_2019, ascending=False).head(10)
plt.figure(figsize=(8, 5))
sns.barplot(x='region', y=col_2019, data=top10_2019, palette='magma')
plt.xticks(rotation=45, ha='right')
plt.title('Топ-10 регіонів за народжуваністю у 2019')
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
sns.histplot(birth_df[col_2019], bins=10, kde=True, color='teal')
plt.title('Розподіл коефіцієнта народжуваності у 2019')
plt.xlabel('Коефіцієнт')
plt.tight_layout()
plt.show()

year_cols = [c for c in birth_df.columns if c != 'region']
year_nums = [int(str(c)[:4]) for c in year_cols]
avg_by_year = birth_df[year_cols].mean()

plt.figure(figsize=(10, 5))
plt.plot(year_nums, avg_by_year.values, marker='o')
plt.title('Середня народжуваність по Україні за роками')
plt.xlabel('Рік')
plt.ylabel('Середній коефіцієнт')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

sample_year_cols = year_cols[-10:] if len(year_cols) > 10 else year_cols
plt.figure(figsize=(10, 5))
sns.boxplot(data=birth_df[sample_year_cols])
plt.xticks(rotation=45)
plt.title('Розподіл народжуваності по вибраних роках')
plt.tight_layout()
plt.show()